In [ ]:
import os
import arcpy
import psycopg2
import subprocess
from datetime import datetime

# =========================
# CONFIGURATION
# =========================

# Source File Geodatabase and layer
FGDB_PATH = r""
LAYER_NAME = "city_boundaries"

# Local staging geodatabase (used for preprocessing)
STAGING_GDB = r"G:\.\.\.\Postgre Data Loading\Postgre Data Loading\staging.gdb"

# Output table name and schema in PostgreSQL
OUTPUT_LAYER = "city_boundaries"
SCHEMA = "base_data"

# Target spatial reference (NAD 1983 StatePlane Tennessee FIPS 4100 US Feet)
SRID = 2274

# PostgreSQL connection (IMPORTANT: use VM IP, not localhost)
PG_CONN = {
    "host": "",
    "dbname": "",
    "user": "",
    "password": "",
    "port": ""
}

# Path to ogr2ogr executable (GDAL)
# Path location may vary by installation
OGR2OGR_PATH = r"C:\.\.\AppData\Local\Programs\OSGeo4W\bin\ogr2ogr.exe"

# Ensure GDAL can find its projection/data files
# Path location may vary by installation
os.environ["GDAL_DATA"] = r"C:\.\.\AppData\Local\Programs\OSGeo4W\share\gdal"

In [ ]:
"""
GNRC PostGIS Data Loading Pipeline
---------------------------------

Purpose:
This script automates the extraction, transformation, and loading (ETL) of
ArcGIS File Geodatabase feature classes into a PostgreSQL/PostGIS database.

Current Implementation:
- Source: File Geodatabase (Boundaries.gdb)
- Layer: City_Boundaries
- Target: PostgreSQL schema (base_data)

Workflow Overview:
1. Data Preparation (ArcPy)
   - Verifies and enforces projection (SRID 2274)
   - Repairs invalid geometry
   - Copies data into a staging geodatabase
   - Standardizes selected attribute fields (e.g., NAME → name)

2. Data Load (GDAL / ogr2ogr)
   - Uses ogr2ogr to load the feature class into PostGIS
   - Converts geometry to MultiPolygon
   - Writes to specified schema and table
   - Overwrites existing table if present

3. PostGIS Cleanup (psycopg2)
   - Ensures target schema exists
   - Enforces geometry type and SRID
   - Creates spatial index (GIST)
   - Repairs invalid geometries using ST_MakeValid

Key Design Decisions:
- MultiPolygon is enforced to avoid geometry type conflicts
- SRID 2274 (NAD 1983 StatePlane Tennessee FIPS 4100 US Feet) is used as the standard
- GDAL (ogr2ogr) is used for reliable and performant data transfer
- ArcPy is used for preprocessing due to strong FGDB support

Requirements:
- ArcGIS Pro / ArcPy environment
- GDAL installed via OSGeo4W (ogr2ogr available)
- PostgreSQL with PostGIS enabled
- Network access to PostgreSQL host (e.g., 10.10.0.130)

Outputs:
- Table created/overwritten in: base_data.city_boundaries
- Geometry stored as MultiPolygon with SRID 2274
- Spatial index created for performance

Notes:
- This script is currently configured for a single layer but is designed
  to be extended into a multi-layer ingestion pipeline.
- Additional schema cleanup (field pruning, naming conventions) can be added
  in future iterations.
""

# =========================
# STEP 1 — DATA PREPARATION (ARCPY)
# =========================

"""
def prep_data():
    """
    Prepares the feature class before loading to PostGIS:
    - Enforces projection
    - Repairs geometry
    - Copies to staging geodatabase
    - Standardizes key fields
    """

    print("Preparing data in ArcGIS Pro...")

    arcpy.env.workspace = FGDB_PATH

    input_fc = os.path.join(FGDB_PATH, LAYER_NAME)
    output_fc = os.path.join(STAGING_GDB, OUTPUT_LAYER)

    target_sr = arcpy.SpatialReference(SRID)

    # Remove existing staged dataset
    if arcpy.Exists(output_fc):
        arcpy.Delete_management(output_fc)

    # Check current spatial reference
    desc = arcpy.Describe(input_fc)
    current_sr = desc.spatialReference
    print(f"Current SR: {current_sr.name}")

    # Reproject if needed
    if current_sr.factoryCode != SRID:
        print("Reprojecting to target coordinate system...")

        projected = os.path.join(STAGING_GDB, "proj_temp")

        if arcpy.Exists(projected):
            arcpy.Delete_management(projected)

        arcpy.management.Project(input_fc, projected, target_sr)
        working_fc = projected
    else:
        print("Already in correct projection")
        working_fc = input_fc

    # Repair geometry issues
    arcpy.management.RepairGeometry(working_fc)

    # Copy to staging
    arcpy.management.CopyFeatures(working_fc, output_fc)

    # Standardize fields (optional but useful)
    fields = [f.name for f in arcpy.ListFields(output_fc)]

    if "NAME" in fields and "name" not in fields:
        arcpy.management.AddField(output_fc, "name", "TEXT")
        arcpy.management.CalculateField(output_fc, "name", "!NAME!")

    if "SOURCE" in fields and "source" not in fields:
        arcpy.management.AddField(output_fc, "source", "TEXT")
        arcpy.management.CalculateField(output_fc, "source", "!SOURCE!")

    print("Data prep complete")

    return output_fc


# =========================
# STEP 2 — LOAD TO POSTGIS (GDAL / ogr2ogr)
# =========================

def load_to_postgis():
    """
    Uses ogr2ogr to load the File Geodatabase layer into PostgreSQL/PostGIS.
    """

    print("Loading to PostGIS...")

    # Build GDAL PostgreSQL connection string
    pg_string = (
        f"PG:host={PG_CONN['host']} "
        f"dbname={PG_CONN['dbname']} "
        f"user={PG_CONN['user']} "
        f"password={PG_CONN['password']} "
        f"port={PG_CONN['port']}"
    )

    # ogr2ogr command
    cmd = [
        OGR2OGR_PATH,
        "-f", "PostgreSQL",
        pg_string,
        FGDB_PATH,        # Source geodatabase
        LAYER_NAME,       # Layer inside the geodatabase
        "-nln", f"{SCHEMA}.{OUTPUT_LAYER}",
        "-nlt", "MULTIPOLYGON",
        "-lco", "GEOMETRY_NAME=geom",
        "-overwrite"
    ]

    print("Running command:")
    print(" ".join(cmd))

    # Execute command and capture output
    result = subprocess.run(cmd, capture_output=True, text=True)

    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)

    # Raise error if command failed
    result.check_returncode()

    print("Load complete")


# =========================
# STEP 3 — POSTGIS CLEANUP
# =========================

def postgis_cleanup():
    """
    Post-load cleanup in PostgreSQL:
    - Ensures schema exists
    - Enforces geometry type and SRID
    - Creates spatial index
    - Repairs invalid geometries
    """

    print("Running PostGIS cleanup...")

    conn = psycopg2.connect(**PG_CONN)
    cur = conn.cursor()

    # Ensure schema exists
    cur.execute(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA};")

    # Enforce geometry type and SRID
    cur.execute(f"""
        ALTER TABLE {SCHEMA}.{OUTPUT_LAYER}
        ALTER COLUMN geom TYPE geometry(MultiPolygon, {SRID})
        USING ST_Multi(
            CASE
                WHEN ST_SRID(geom) != {SRID}
                THEN ST_Transform(geom, {SRID})
                ELSE geom
            END
        );
    """)

    # Create spatial index
    cur.execute(f"""
        CREATE INDEX IF NOT EXISTS {OUTPUT_LAYER}_geom_idx
        ON {SCHEMA}.{OUTPUT_LAYER}
        USING GIST (geom);
    """)

    # Fix invalid geometries
    cur.execute(f"""
        UPDATE {SCHEMA}.{OUTPUT_LAYER}
        SET geom = ST_MakeValid(geom)
        WHERE NOT ST_IsValid(geom);
    """)

    conn.commit()

    # Verify SRID
    cur.execute(f"SELECT DISTINCT ST_SRID(geom) FROM {SCHEMA}.{OUTPUT_LAYER};")
    srids = cur.fetchall()
    print("SRID check:", srids)

    cur.close()
    conn.close()

    print("PostGIS cleanup complete")


# =========================
# MAIN PIPELINE
# =========================

def run_pipeline():
    """
    Runs the full ETL pipeline:
    1. Prepare data in ArcGIS
    2. Load to PostGIS
    3. Clean and standardize in PostGIS
    """

    start = datetime.now()
    print(f"Starting pipeline: {start}")

    prep_data()
    load_to_postgis()
    postgis_cleanup()

    end = datetime.now()
    print(f"Pipeline complete: {end}")
    print(f"Duration: {end - start}")


if __name__ == "__main__":
    run_pipeline()

In [ ]:
"""
GNRC PostGIS Data Loading Pipeline
---------------------------------

Purpose:
This script automates the extraction, transformation, and loading (ETL) of
ArcGIS File Geodatabase feature classes into a PostgreSQL/PostGIS database.

Current Implementation:
- Source: File Geodatabase (Boundaries.gdb)
- Layer: City_Boundaries
- Target: PostgreSQL database (gnrc_prod) schema (base_data)

Workflow Overview:
1. Data Preparation (ArcPy)
   - Verifies and enforces projection (SRID 2274)
   - Repairs invalid geometry
   - Copies data into a staging geodatabase
   - Standardizes selected attribute fields (e.g., NAME → name)

2. Data Load (GDAL / ogr2ogr)
   - Uses ogr2ogr to load the feature class into PostGIS
   - Converts geometry to MultiPolygon
   - Writes to specified schema and table
   - Overwrites existing table if present

3. PostGIS Cleanup (psycopg2)
   - Ensures target schema exists
   - Removes duplicate OBJECTID fields created during GDAL load (e.g., objectid_1, objectid_2)
   - Renames original OBJECTID → esri_objectid for reference
   - Detects and drops any existing primary key created by GDAL
   - Creates a clean, consistent primary key (id SERIAL)
   - Enforces geometry type and SRID (MultiPolygon, 2274)
   - Creates spatial index (GIST)
   - Repairs invalid geometries using ST_MakeValid

Key Design Decisions:
- MultiPolygon is enforced to avoid geometry type conflicts
- SRID 2274 (NAD 1983 StatePlane Tennessee FIPS 4100 US Feet) is used as the standard
- GDAL (ogr2ogr) is used for reliable and performant data transfer
- ArcPy is used for preprocessing due to strong FGDB support
- Database-managed primary key (id) is used instead of relying on ArcGIS OBJECTID
- Existing primary keys are dropped to ensure consistent schema across all datasets

Resulting Table Structure:
- id                → Primary key (system-managed, consistent across all layers)
- geom              → Geometry (MultiPolygon, SRID 2274)
- esri_objectid     → Original ArcGIS OBJECTID (preserved for reference)
- other fields      → Cleaned attribute data

Requirements:
- ArcGIS Pro / ArcPy environment
- GDAL installed via OSGeo4W (ogr2ogr available)
- PostgreSQL with PostGIS enabled
- Network access to PostgreSQL host (e.g., 10.10.0.130)

Outputs:
- Table created/overwritten in: base_data.city_boundaries
- Geometry stored as MultiPolygon with SRID 2274
- Spatial index created for performance
- Clean, consistent primary key structure across datasets

Notes:
- This script is currently configured for a single layer but is designed
  to be extended into a multi-layer ingestion pipeline.
- The pipeline intentionally normalizes schema after load rather than relying on GDAL defaults.
- Query Layers in ArcGIS should use: id AS objectid for stability.
- Large multipart geometries may trigger GDAL warnings but do not affect correctness.
"""


# =========================
# STEP 1 — DATA PREP
# =========================

def prep_data():
    print("Preparing data in ArcGIS Pro...")

    arcpy.env.workspace = FGDB_PATH

    input_fc = os.path.join(FGDB_PATH, LAYER_NAME)
    output_fc = os.path.join(STAGING_GDB, OUTPUT_LAYER)

    target_sr = arcpy.SpatialReference(SRID)

    if arcpy.Exists(output_fc):
        arcpy.Delete_management(output_fc)

    desc = arcpy.Describe(input_fc)
    current_sr = desc.spatialReference

    print(f"Current SR: {current_sr.name}")

    if current_sr.factoryCode != SRID:
        projected = os.path.join(STAGING_GDB, "proj_temp")

        if arcpy.Exists(projected):
            arcpy.Delete_management(projected)

        arcpy.management.Project(input_fc, projected, target_sr)
        working_fc = projected
    else:
        working_fc = input_fc

    arcpy.management.RepairGeometry(working_fc)
    arcpy.management.CopyFeatures(working_fc, output_fc)

    print("Data prep complete")


# =========================
# STEP 2 — LOAD TO POSTGIS
# =========================

def load_to_postgis():
    print("Loading to PostGIS...")

    pg_string = (
        f"PG:host={PG_CONN['host']} "
        f"dbname={PG_CONN['dbname']} "
        f"user={PG_CONN['user']} "
        f"password={PG_CONN['password']} "
        f"port={PG_CONN['port']}"
    )

    cmd = [
        OGR2OGR_PATH,
        "-f", "PostgreSQL",
        pg_string,
        FGDB_PATH,
        LAYER_NAME,
        "-nln", f"{SCHEMA}.{OUTPUT_LAYER}",
        "-nlt", "MULTIPOLYGON",
        "-lco", "GEOMETRY_NAME=geom",
        "-overwrite"
    ]

    print("Running command:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)

    result.check_returncode()

    print("Load complete")


# =========================
# STEP 3 — POSTGIS CLEANUP (FIXED)
# =========================

def postgis_cleanup():
    print("Running PostGIS cleanup...")

    conn = psycopg2.connect(**PG_CONN)
    cur = conn.cursor()

    cur.execute(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA};")

    # -------------------------
    # DROP duplicate objectid fields
    # -------------------------
    cur.execute(f"""
        DO $$
        DECLARE col RECORD;
        BEGIN
            FOR col IN
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = '{SCHEMA}'
                AND table_name = '{OUTPUT_LAYER}'
                AND column_name LIKE 'objectid_%'
            LOOP
                EXECUTE format(
                    'ALTER TABLE {SCHEMA}.{OUTPUT_LAYER} DROP COLUMN %I',
                    col.column_name
                );
            END LOOP;
        END$$;
    """)

    # -------------------------
    # RENAME original objectid
    # -------------------------
    cur.execute(f"""
        DO $$
        BEGIN
            IF EXISTS (
                SELECT 1 FROM information_schema.columns
                WHERE table_schema = '{SCHEMA}'
                AND table_name = '{OUTPUT_LAYER}'
                AND column_name = 'objectid'
            ) THEN
                ALTER TABLE {SCHEMA}.{OUTPUT_LAYER}
                RENAME COLUMN objectid TO esri_objectid;
            END IF;
        END$$;
    """)

    # -------------------------
    # DROP existing PRIMARY KEY (THIS FIXES YOUR ERROR)
    # -------------------------
    cur.execute(f"""
        DO $$
        DECLARE pk_name text;
        BEGIN
            SELECT constraint_name INTO pk_name
            FROM information_schema.table_constraints
            WHERE table_schema = '{SCHEMA}'
            AND table_name = '{OUTPUT_LAYER}'
            AND constraint_type = 'PRIMARY KEY';

            IF pk_name IS NOT NULL THEN
                EXECUTE format(
                    'ALTER TABLE {SCHEMA}.{OUTPUT_LAYER} DROP CONSTRAINT %I',
                    pk_name
                );
            END IF;
        END$$;
    """)

    # -------------------------
    # CREATE CLEAN PRIMARY KEY
    # -------------------------
    cur.execute(f"""
        ALTER TABLE {SCHEMA}.{OUTPUT_LAYER}
        ADD COLUMN id SERIAL;
    """)

    cur.execute(f"""
        ALTER TABLE {SCHEMA}.{OUTPUT_LAYER}
        ADD PRIMARY KEY (id);
    """)

    # -------------------------
    # GEOMETRY STANDARDIZATION
    # -------------------------
    cur.execute(f"""
        ALTER TABLE {SCHEMA}.{OUTPUT_LAYER}
        ALTER COLUMN geom TYPE geometry(MultiPolygon, {SRID})
        USING ST_Multi(
            CASE
                WHEN ST_SRID(geom) != {SRID}
                THEN ST_Transform(geom, {SRID})
                ELSE geom
            END
        );
    """)

    # Spatial index
    cur.execute(f"""
        CREATE INDEX IF NOT EXISTS {OUTPUT_LAYER}_geom_idx
        ON {SCHEMA}.{OUTPUT_LAYER}
        USING GIST (geom);
    """)

    # Fix invalid geometries
    cur.execute(f"""
        UPDATE {SCHEMA}.{OUTPUT_LAYER}
        SET geom = ST_MakeValid(geom)
        WHERE NOT ST_IsValid(geom);
    """)

    conn.commit()

    cur.execute(f"SELECT DISTINCT ST_SRID(geom) FROM {SCHEMA}.{OUTPUT_LAYER};")
    print("SRID check:", cur.fetchall())

    cur.close()
    conn.close()

    print("PostGIS cleanup complete")


# =========================
# MAIN
# =========================

def run_pipeline():
    start = datetime.now()
    print(f"Starting pipeline: {start}")

    prep_data()
    load_to_postgis()
    postgis_cleanup()

    end = datetime.now()
    print(f"Pipeline complete: {end}")
    print(f"Duration: {end - start}")


if __name__ == "__main__":
    run_pipeline()